## Token & Param

In [3]:
def get_active_params(
    n_layers: int,
    dmodel: int,
    expert_hidden_size: int,
    n_experts: int,
    vocab_size: int = 50257,
    with_embeddings: bool = False
):
    # -- 1) MHA per layer (same as total, because all attention heads/layers are used)
    attention_params_per_layer = 4 * (dmodel ** 2)

    # -- 2) Gating network per layer: we must compute gating for all experts
    gating_params_per_layer = (dmodel * n_experts)

    # -- 3) Only 1 expert is active per layer (top-1 gating):
    ff_moe_params_per_layer = 3 * dmodel * expert_hidden_size

    layer_params = attention_params_per_layer + ff_moe_params_per_layer
    transformer_params = n_layers * layer_params

    # -- 4) Embeddings (input + output).
    embedding_params = 2 * vocab_size * dmodel

    active_params = transformer_params

    if with_embeddings:
      active_params += embedding_params

    return active_params

In [6]:
k = 24
n_experts = 32

vocab_size = 50257
dhead = 64
dmodel = dhead * k
print(f'dmodel: {dmodel}')
n_blocks = k

tokens = 512

params = get_active_params(
    n_layers=k,
    dmodel=dmodel,
    expert_hidden_size=3 * dmodel,
    n_experts=n_experts,
    vocab_size=vocab_size,
    with_embeddings=True
)
enc_params = get_active_params(
    n_layers=k,
    dmodel=dmodel,
    expert_hidden_size=3 * dmodel,
    n_experts=n_experts,
    vocab_size=vocab_size,
    with_embeddings=False
)
print(f'params: {params / 1e6:.0f}M\t encoder: {enc_params/1e6:.0f}M\tembedding: {(params - enc_params)/2e6:.0f}M')
flops_forward = 2 * tokens * params
theoretical_flops = 6 * tokens * params
print(f'flops_forward: {flops_forward / 512e6:.1f} MFLOPS per token')

dmodel: 1536
params: 890M	 encoder: 736M	embedding: 77M
flops_forward: 1781.0 MFLOPS per token


In [2]:
tokens = 512
# tokens = 6e4 * 512 * 512
if tokens is None:
    tokens = 20 * params

ratio = tokens / params
print(f'tokens: {tokens / 1e9:.2f}B')
print(f'ratio: {ratio}')
print(5e4)

tokens: 0.00B
ratio: 2.0275092454421593e-06
50000.0


In [3]:
batch_size = 256
seq_len = 256
n_steps = tokens / (batch_size * seq_len)
print(f'n steps: {n_steps:.0f}')

n steps: 0


In [4]:
float_bytes = 4
tokens_memory = float_bytes * dmodel * batch_size * seq_len
print(f'batch_memory: {tokens_memory / 1e6:.0f} MB')

model_memory = 4 * float_bytes * params
print(f'model_memory: {model_memory / 1e9:.0f} GB')

batch_memory: 268 MB
model_memory: 4 GB


## compute cost $$$

In [5]:
mfu = 0.3
gpu_flops = 800 * 1e12

In [6]:
flops_forward = 2 * tokens * params
theoretical_flops = 6 * tokens * params
print(f'flops_forward: {flops_forward / 1e12:.4f} TFLOPS')
print(f'theoretical_flops: {theoretical_flops / 1e18:.2f}*1E6 TFLOPS')
gpu_hours = theoretical_flops / (gpu_flops * 3600 * mfu)
print(f'GPU hours: {gpu_hours:.1f}')
n_gpus = 4
print(f'training hours: {gpu_hours / n_gpus:.1f}')

flops_forward: 0.2586 TFLOPS
theoretical_flops: 0.00*1E6 TFLOPS
GPU hours: 0.0
training hours: 0.0


In [40]:
grid_size = 15
total_gpu_hours = grid_size * gpu_hours
print(f'total_gpu_hours: {total_gpu_hours:.0f}')

total_gpu_hours: 6302


## Calculate MFU

In [92]:
minutes = 28.25
steps = 236
batch_size = 512
seq_len = 512
vocab_size = 50000
k = 48
gpu_flops = 800 * 1e12
n_gpus = 4
dmodel = dhead * k
print(f'dmodel: {dmodel}')
n_blocks = k

params = n_blocks * 12 * dmodel ** 2 + 2 * dmodel * vocab_size
print(f'params: {params / 1e6:.0f}M')

dmodel: 3072
params: 5743M


In [93]:
tokens_processed = steps * batch_size * seq_len
th_flops = 6 * tokens_processed * params
real_flops = gpu_flops * n_gpus * 60 * minutes
mfu = th_flops / real_flops
print(f'MFU: {mfu:.3f}')

MFU: 0.393
